In [1]:
import os
import numpy as np
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
import torch_geometric.transforms as T
from scipy.spatial import cKDTree
from dataset.cell_motion import CellMotionRollout
from model.simulator import Simulator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

dataset_dir = os.path.join(os.getcwd(), 'dataset')
load_dir = 'checkpoint/20260204_161356/best_epoch/best_checkpoint.pth'
sequence_length = 20
batch_size = 1
max_rollout_steps = 80
time_step = 1.0
edge_k = 8
velocity_blend = 0.25
max_step = 2.5
correction_interval = 20
correction_blend = 0.3
save_dir = 'long_horizon_outputs'
os.makedirs(save_dir, exist_ok=True)

Using device: cuda


In [2]:
def compute_velocity_features(flat_history: torch.Tensor, sequence_len: int):
    time_feature = flat_history[:, -1:]
    pos_flat = flat_history[:, :-1]
    num_nodes = pos_flat.size(0)
    pos_seq = pos_flat.view(num_nodes, sequence_len, 2)
    velocities = pos_seq[:, 1:, :] - pos_seq[:, :-1, :]
    zeros = torch.zeros(num_nodes, 1, 2, device=flat_history.device)
    velocities = torch.cat([zeros, velocities], dim=1).view(num_nodes, -1)
    return torch.cat([velocities, time_feature], dim=1)


def build_edges_from_geometry(pos_np: np.ndarray, k: int = 8):
    k = max(1, min(k, pos_np.shape[0] - 1))
    tree = cKDTree(pos_np)
    distances, indices = tree.query(pos_np, k=k + 1)
    edge_index = []
    edge_attr = []
    for i in range(pos_np.shape[0]):
        for j in range(1, indices.shape[1]):
            nbr = indices[i, j]
            edge_index.append([i, nbr])
            delta = pos_np[nbr] - pos_np[i]
            edge_attr.append([distances[i, j], delta[0], delta[1]])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32)
    return edge_index, edge_attr


def rollout(model: torch.nn.Module, simulator, frames, rollout_steps: int):
    if len(frames) == 0:
        return np.zeros((0, 0, 2))

    model.eval()
    with torch.no_grad():  # ★ 修复1：禁止构建计算图，显存不再随步数累积
        # ★ 修复2：第0帧按需搬到 GPU
        first_frame = frames[0].to(device)
        num_nodes = first_frame.pos.size(0)
        history = first_frame.x[:, :sequence_length * 2].view(num_nodes, sequence_length, 2)
        time_col = first_frame.x[:, -1:]
        current_pos = first_frame.pos
        last_disp = current_pos - history[:, -1]
        # ★ 修复3：预测结果立即转回 CPU，不在 GPU 上堆叠
        predictions = [current_pos.cpu().numpy()]

        for step in range(min(rollout_steps, len(frames) - 1)):
            edge_index, edge_attr = build_edges_from_geometry(current_pos.cpu().numpy(), k=edge_k)
            edge_index = edge_index.to(device)
            edge_attr = edge_attr.to(device)

            vel_input = compute_velocity_features(
                torch.cat([history.reshape(num_nodes, -1), time_col], dim=1),
                sequence_length
            )
            g = Data(x=vel_input, pos=current_pos, edge_index=edge_index, edge_attr=edge_attr)
            pred_norm = model(g)
            pred_disp = simulator._output_normalizer.inverse(pred_norm)
            pred_disp = torch.clamp(pred_disp, -max_step, max_step)
            blended_disp = (1.0 - velocity_blend) * pred_disp + velocity_blend * last_disp
            next_pos_pred = current_pos + blended_disp

            use_corr = (
                correction_interval
                and (step + 1) % correction_interval == 0
                and (step + 1) < len(frames)
            )
            if use_corr:
                # ★ 修复2（续）：ground truth 帧按需搬到 GPU
                gt_pos = frames[step + 1].pos.to(device)
                next_pos = gt_pos * correction_blend + next_pos_pred * (1.0 - correction_blend)
            else:
                next_pos = next_pos_pred

            # ★ 修复3（续）：结果立即回 CPU
            predictions.append(next_pos.cpu().numpy())
            history = torch.cat([history[:, 1:], next_pos.unsqueeze(1)], dim=1)
            time_col = time_col + time_step
            last_disp = next_pos - current_pos
            current_pos = next_pos

            # ★ 修复4：每隔10步主动释放 GPU 缓存碎片
            if (step + 1) % 10 == 0:
                torch.cuda.empty_cache()

    return np.stack(predictions, axis=0)

In [4]:
transformer = T.Compose([
    T.Cartesian(norm=False),
    T.Distance(norm=False)
])

simulator = Simulator(
    message_passing_num=15,
    node_input_size=sequence_length * 2 + 1,
    edge_input_size=3,
    device=device,
    hidden_size=256,
    model_dir=load_dir
)
simulator.load_checkpoint(load_dir)
model = simulator.model.to(device)
model.eval()

dataset = CellMotionRollout(
    dataset_dir=dataset_dir,
    split='test_e0n300',
    sequence_length=sequence_length
)
loader = DataLoader(dataset, batch_size=batch_size, num_workers=0)

frames = []
for batch in loader:
    batch = transformer(batch)
    frames.append(batch)  # 不提前 .to(device)

predicted = rollout(model, simulator, frames, max_rollout_steps)

# ★ 修复：统一加 .cpu() 再转 numpy，避免 tensor 在 GPU 上报错
gt = np.stack([f.pos.cpu().numpy() for f in frames[:predicted.shape[0]]], axis=0)
rmse = np.sqrt(np.mean((predicted - gt) ** 2)) if predicted.size > 0 else np.nan
print(f'Predicted trajectory shape: {predicted.shape}')
print(f'RMSE over {predicted.shape[0]-1} rollout steps: {rmse:.4f}')

np.save(os.path.join(save_dir, 'long_rollout_pred.npy'), predicted)
np.save(os.path.join(save_dir, 'long_rollout_gt.npy'), gt)

Simulator model initialized with hidden_size=256
Simulator model loaded checkpoint checkpoint/20260204_161356/best_epoch/best_checkpoint.pth
Predicted trajectory shape: (81, 389, 2)
RMSE over 80 rollout steps: 1.8435
